# CHL7201 / CLL788 — Assignment 2
RNN, encoder–decoder, attention and VAE

Numerical answers for Q1–Q4 are computed below and checked with finite differences.
Q5 trains the flotation-plant models on the Kaggle mining dataset.
The same Adam optimiser, hidden size 32, MSE loss and gradient-norm clip of 1 are used inside each part.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
torch.set_num_threads(4)
torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

## Q1. Scalar RNN: forward pass, BPTT and one gradient step

Loss uses only the last two steps, $E = E_2 + E_3$.
$h_0 = 0.2$ is not zero, so the $t = 1$ term in $\partial E/\partial W_{hh}$ is kept.

In [ ]:
Wxh, Whh, Why = 0.8, 0.6, 1.2
bh, by = 0.1, 0.0
h0 = 0.2
xs = np.array([0.5, -0.2, 0.9])
ds = np.array([0.4, 0.3, 0.8])

hs = [h0]
zs, ht, tp, ys, Es = [], [], [], [], []
h = h0
for t in range(3):
    z = Wxh * xs[t] + Whh * h + bh
    h = math.tanh(z)
    y = Why * h + by
    E = (ds[t] - y) ** 2
    zs.append(z); ht.append(h); tp.append(1 - h ** 2); ys.append(y); Es.append(E)
    hs.append(h)

print("Forward pass")
print(f"{'t':>2} {'z':>8} {'h':>8} {'tanh\'':>8} {'y':>8} {'E':>8}")
for t in range(3):
    print(f"{t+1:2d} {zs[t]:8.4f} {ht[t]:8.4f} {tp[t]:8.4f} {ys[t]:8.4f} {Es[t]:8.4f}")
E_tot = Es[1] + Es[2]
print(f"E = E2 + E3 = {E_tot:.4f}")

# dE_t/dy_t = -2(d_t - y_t). Only t = 2, 3 are in the loss.
dEdy = np.zeros(3)
dEdy[1] = -2 * (ds[1] - ys[1])
dEdy[2] = -2 * (ds[2] - ys[2])
g_Why = dEdy[1] * ht[1] + dEdy[2] * ht[2]
print(f"\ndE2/dWhy = {dEdy[1]:.4f} * {ht[1]:.4f} = {dEdy[1]*ht[1]:.4f}")
print(f"dE3/dWhy = {dEdy[2]:.4f} * {ht[2]:.4f} = {dEdy[2]*ht[2]:.4f}")
print(f"dE/dWhy  = {g_Why:.4f}")

def chain(T, i):
    # dh_T/dh_i , indices 0-based. Product of tanh'(z_j) * Whh for j = i+1..T
    val = 1.0
    for j in range(i + 1, T + 1):
        val *= tp[j] * Whh
    return val

g_Whh = g_Wxh = 0.0
print("\nBPTT terms. chain = dh_T/dh_i, local = dh_i/dW with h_{i-1} held fixed")
for T in (1, 2):
    dEdh = dEdy[T] * Why
    print(f"\nFrom E{T+1}: dE/dh{T+1} = {dEdh:.4f}")
    print(f"{'i':>2} {'chain':>8} {'dhi/dWhh':>10} {'dhi/dWxh':>10} {'Whh term':>10} {'Wxh term':>10}")
    for i in range(T + 1):
        ch = chain(T, i)
        h_prev = h0 if i == 0 else ht[i - 1]
        loc_hh = tp[i] * h_prev
        loc_xh = tp[i] * xs[i]
        term_hh = dEdh * ch * loc_hh
        term_xh = dEdh * ch * loc_xh
        g_Whh += term_hh
        g_Wxh += term_xh
        print(f"{i+1:2d} {ch:8.4f} {loc_hh:10.4f} {loc_xh:10.4f} {term_hh:10.4f} {term_xh:10.4f}")

print(f"\ndE/dWhh = {g_Whh:.4f}")
print(f"dE/dWxh = {g_Wxh:.4f}")
print(f"dh3/dh1 = {chain(2, 0):.4f}")

def loss(wxh, whh, why):
    h = h0
    total = 0.0
    for t in range(3):
        h = math.tanh(wxh * xs[t] + whh * h + bh)
        y = why * h + by
        if t >= 1:
            total += (ds[t] - y) ** 2
    return total

eps = 1e-6
fd_why = (loss(Wxh, Whh, Why + eps) - loss(Wxh, Whh, Why - eps)) / (2 * eps)
fd_whh = (loss(Wxh, Whh + eps, Why) - loss(Wxh, Whh - eps, Why)) / (2 * eps)
fd_wxh = (loss(Wxh + eps, Whh, Why) - loss(Wxh - eps, Whh, Why)) / (2 * eps)
assert abs(fd_why - g_Why) < 1e-6
assert abs(fd_whh - g_Whh) < 1e-6
assert abs(fd_wxh - g_Wxh) < 1e-6
print("Finite-difference check passed.")

eta = 0.5
Wxh2 = Wxh - eta * g_Wxh
Whh2 = Whh - eta * g_Whh
Why2 = Why - eta * g_Why
print(f"\nUpdated weights, eta = 0.5")
print(f"Wxh {Wxh:.4f} -> {Wxh2:.4f}")
print(f"Whh {Whh:.4f} -> {Whh2:.4f}")
print(f"Why {Why:.4f} -> {Why2:.4f}")
E_new = loss(Wxh2, Whh2, Why2)
print(f"New E = {E_new:.4f}   old E = {E_tot:.4f}   decreased = {E_new < E_tot}")

print("\nForward pass with the new weights")
h = h0
for t in range(3):
    z = Wxh2 * xs[t] + Whh2 * h + bh
    h = math.tanh(z)
    y = Why2 * h + by
    print(f"t={t+1}  z={z:.4f}  h={h:.4f}  y={y:.4f}  E={(ds[t]-y)**2:.4f}")

bound = abs(Whh) ** 49
gru = (1 - 0.1) ** 49
print(f"\nLargest |dh50/dh1| <= |Whh|^49 = {bound:.4e}")
print(f"GRU additive path (1 - 0.1)^49 = {gru:.4e}")

## Q2. Encoder–decoder: teacher forcing and inference

Encoder: $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$, with $c = h_3$ and $h_0 = 0$.

Decoder: $s_t = \tanh(W_s s_{t-1} + W_y e(y_{t-1}) + W_c c)$, then a 3-way softmax.
Rows are ordered (Normal, Alarm, END).

In [ ]:
Wx = np.array([0.5, -0.4])
Wh = np.array([[0.4, 0.1], [-0.2, 0.3]])
b = np.array([0.0, 0.1])
Ws = np.eye(2) * 0.5
Wy = np.array([[-1.0, -1.0], [-1.0, 0.5]])
Wc = np.eye(2)
Wout = np.array([[1.5, -1.0], [-1.0, 2.0], [-1.0, -1.0]])
bout = np.array([0.8, 0.0, -0.5])
labels = ["Normal", "Alarm", "END"]
emb = {
    "START": np.array([0.0, 0.0]),
    "Normal": np.array([1.0, 0.0]),
    "Alarm": np.array([0.0, 1.0]),
}

def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()

def show_probs(title, s, logits, p, yhat):
    print(title)
    print("  s     ", np.round(s, 4))
    print("  logits", np.round(logits, 4))
    print("  P     ", np.round(p, 4), " ->", yhat)

h = np.zeros(2)
H_enc = []
for t, x in enumerate([1.0, 0.5, -1.0], start=1):
    z = Wx * x + Wh @ h + b
    h = np.tanh(z)
    H_enc.append(h.copy())
    print(f"h{t} = {np.round(h, 4)}")
H_enc = np.stack(H_enc)
c = H_enc[-1].copy()
print("c = h3 =", np.round(c, 4))

def decoder_step(s_prev, token, context):
    s = np.tanh(Ws @ s_prev + Wy @ emb[token] + Wc @ context)
    logits = Wout @ s + bout
    p = softmax(logits)
    return s, logits, p, labels[int(p.argmax())]

s1, logits1, p1, y1 = decoder_step(c, "START", c)
show_probs("Step 1, input START", s1, logits1, p1, y1)
print("True y1 is Normal, so the prediction is", "right" if y1 == "Normal" else "wrong")

s2_tf, l2_tf, p2_tf, y2_tf = decoder_step(s1, "Normal", c)
show_probs("Step 2, teacher forcing (true Normal)", s2_tf, l2_tf, p2_tf, y2_tf)

s2_inf, l2_inf, p2_inf, y2_inf = decoder_step(s1, y1, c)
show_probs(f"Step 2, inference (feed {y1})", s2_inf, l2_inf, p2_inf, y2_inf)
print(f"Inference sequence, stop at END or after 2 steps: ({y1}, {y2_inf})")

## Q3. Attention encoder–decoder

Same encoder states and decoder weights. At each step

$$e_{ti} = s_{t-1}^{\top} h_i,\quad
\alpha_{ti} = \mathrm{softmax}(e_t),\quad
c_t = \sum_i \alpha_{ti} h_i.$$

In [ ]:
def attn_step(s_prev, token):
    scores = H_enc @ s_prev
    alpha = softmax(scores)
    context = alpha @ H_enc
    s = np.tanh(Ws @ s_prev + Wy @ emb[token] + Wc @ context)
    logits = Wout @ s + bout
    p = softmax(logits)
    return scores, alpha, context, s, logits, p, labels[int(p.argmax())]

e1, a1, c1, s1a, logits1a, p1a, y1a = attn_step(c, "START")
print("Step 1")
print("  e1", np.round(e1, 4))
print("  a1", np.round(a1, 4), " most weight on h" + str(int(a1.argmax()) + 1))
print("  c1", np.round(c1, 4))
show_probs("  decoder", s1a, logits1a, p1a, y1a)

e2, a2, c2, s2a, logits2a, p2a, y2a = attn_step(s1a, "Normal")
print("Step 2, teacher forcing")
print("  e2", np.round(e2, 4))
print("  a2", np.round(a2, 4), " most weight on h" + str(int(a2.argmax()) + 1))
print("  c2", np.round(c2, 4))
show_probs("  decoder", s2a, logits2a, p2a, y2a)
print(f"Predicted sequence: ({y1a}, {y2a})")
print("True sequence is (Normal, END):", y1a == "Normal" and y2a == "END")

c_mean = H_enc.mean(axis=0)
s_mean = np.tanh(Ws @ c + Wy @ emb["START"] + Wc @ c_mean)
p_mean = softmax(Wout @ s_mean + bout)
print("\nMean pooling, alpha = 1/3")
print("  c ", np.round(c_mean, 4))
print("  s1", np.round(s_mean, 4))
print("  P ", np.round(p_mean, 4))
print(f"P(Normal): Q2 last-state {p1[0]:.4f},  attention {p1a[0]:.4f},  mean pool {p_mean[0]:.4f}")

## Q4. Variational autoencoder

$$\mu = w_\mu^{\top} x + b_\mu,\quad
\log\sigma^2 = w_v^{\top} x + b_v,\quad
z = \mu + \sigma\varepsilon,\quad
\hat x = \mathrm{sigmoid}(W' z + b').$$

$$L = \sum_j (x_j - \hat x_j)^2 + \tfrac12(\mu^2 + \sigma^2 - \log\sigma^2 - 1).$$

In [ ]:
def sigmoid(u):
    return 1 / (1 + np.exp(-u))

x = np.array([0.8, 0.4])
w_mu, b_mu = np.array([0.5, 0.5]), 0.0
w_v, b_v = np.array([-1.0, 0.5]), -0.6
Wp, bp = np.array([2.0, 1.0]), np.array([-0.5, -1.0])
eps_noise = 0.5

mu = float(w_mu @ x + b_mu)
logvar = float(w_v @ x + b_v)
var = math.exp(logvar)
sigma = math.sqrt(var)
z = mu + sigma * eps_noise
u = Wp * z + bp
xhat = sigmoid(u)
rec = float(np.sum((x - xhat) ** 2))
kl = 0.5 * (mu ** 2 + var - logvar - 1)
L = rec + kl
print(f"mu = {mu:.4f}")
print(f"log sigma^2 = {logvar:.4f}")
print(f"sigma^2 = {var:.4f}")
print(f"sigma = {sigma:.4f}")
print(f"z = {z:.4f}")
print("u   ", np.round(u, 4))
print("xhat", np.round(xhat, 4))
print(f"reconstruction = {rec:.4f}")
print(f"KL             = {kl:.4f}")
print(f"L              = {L:.4f}")

# dL/dz through the sigmoid decoder, then the reparameterisation
dL_dxh = -2 * (x - xhat)
sig_prime = xhat * (1 - xhat)
dL_du = dL_dxh * sig_prime
dL_dz = float(Wp @ dL_du)
dL_dmu = dL_dz * 1 + mu
dL_dlogvar = dL_dz * (eps_noise * sigma / 2) + 0.5 * (var - 1)
print(f"\ndL/dz       = {dL_dz:.4f}")
print(f"dL/dmu      = {dL_dmu:.4f}")
print(f"dL/dlogvar  = {dL_dlogvar:.4f}")

def L_of(mu_v, logvar_v, eps_v=eps_noise, x_v=x):
    var_v = math.exp(logvar_v)
    z_v = mu_v + math.sqrt(var_v) * eps_v
    xh = sigmoid(Wp * z_v + bp)
    return float(np.sum((x_v - xh) ** 2) + 0.5 * (mu_v ** 2 + var_v - logvar_v - 1))

h = 1e-6
fd_mu = (L_of(mu + h, logvar) - L_of(mu - h, logvar)) / (2 * h)
fd_lv = (L_of(mu, logvar + h) - L_of(mu, logvar - h)) / (2 * h)
assert abs(fd_mu - dL_dmu) < 1e-6
assert abs(fd_lv - dL_dlogvar) < 1e-6
print("Finite-difference check passed.")

def recon_error(sample):
    mu_s = float(w_mu @ sample + b_mu)
    xh = sigmoid(Wp * mu_s + bp)  # epsilon = 0, so z = mu
    err = float(np.sum((sample - xh) ** 2))
    return mu_s, xh, err

xn = np.array([0.8, 0.4])
xf = np.array([0.2, 0.9])
mu_n, xh_n, e_n = recon_error(xn)
mu_f, xh_f, e_f = recon_error(xf)
print(f"\nNormal  mu={mu_n:.4f}  xhat={np.round(xh_n,4)}  error={e_n:.4f}")
print(f"Suspect mu={mu_f:.4f}  xhat={np.round(xh_f,4)}  error={e_f:.4f}")
print(f"Suspect / normal = {e_f / e_n:.4f}")
print("Flagged sample: suspect xf" if e_f > e_n else "Flagged sample: normal x")

z_gen = -1.0
xh_gen = sigmoid(Wp * z_gen + bp)
print("Generated xhat at z = -1:", np.round(xh_gen, 4))

## Q5. Flotation plant

Part A builds one row per hour, drops `% Iron Concentrate`, refuses any window that crosses a gap, and fits the scaler on the first 70% of hours only.

Part B predicts silica at the next hour from the previous $L$ hours of the 21 process variables.
Part C forecasts the next 6 silica values from the previous 24 hours of process variables plus past silica.
Part D uses the lecture dot-product attention. A linear map $W_c$ sends the 32-dimensional context to 8 numbers before it is joined to the previous silica value, so the lab reading is not swamped by the context.

In [ ]:
DATA = Path("/tmp/flotation/MiningProcess_Flotation_Plant_Database.csv")
if not DATA.exists():
    import urllib.request
    DATA.parent.mkdir(parents=True, exist_ok=True)
    url = (
        "https://media.githubusercontent.com/media/purnaspr/"
        "mining_project-ml/main/data/MiningProcess_Flotation_Plant_Database.csv"
    )
    print("downloading", url)
    urllib.request.urlretrieve(url, DATA)

raw = pd.read_csv(DATA, decimal=",")
if raw.shape[1] == 1:
    raw = pd.read_csv(DATA, sep=";", decimal=",")
raw["date"] = pd.to_datetime(raw["date"])
print("raw rows", len(raw), "columns", raw.shape[1])
print("from", raw["date"].min(), "to", raw["date"].max())

silica = "% Silica Concentrate"
raw = raw.drop(columns=["% Iron Concentrate"])
value_cols = [c for c in raw.columns if c != "date"]
hourly = raw.groupby(raw["date"].dt.floor("h"))[value_cols].mean().sort_index()
print("hourly rows", len(hourly), "median raw rows per hour",
      int(raw.groupby(raw["date"].dt.floor("h")).size().median()))

gaps = hourly.index.to_series().diff()
gap_at = gaps[gaps > pd.Timedelta(hours=1)]
print("gaps larger than 1 hour:", len(gap_at))
for ts, delta in gap_at.items():
    print(" ", ts - delta, "->", ts, "jump", delta)

proc_cols = [c for c in hourly.columns if c != silica]
assert len(proc_cols) == 21
segment = (hourly.index.to_series().diff() != pd.Timedelta(hours=1)).cumsum().to_numpy()
n = len(hourly)
n_train, n_val = int(n * 0.70), int(n * 0.15)
n_test = n - n_train - n_val
split = np.array(["train"] * n_train + ["val"] * n_val + ["test"] * n_test)
print(f"split train/val/test = {n_train}/{n_val}/{n_test}")

sx = StandardScaler().fit(hourly.iloc[:n_train][proc_cols])
sy = StandardScaler().fit(hourly.iloc[:n_train][[silica]])
X_proc = sx.transform(hourly[proc_cols]).astype(np.float32)
y_sil = sy.transform(hourly[[silica]]).ravel().astype(np.float32)
y_std = float(sy.scale_[0])
y_mean = float(sy.mean_[0])
print(f"train silica mean {y_mean:.4f}  std {y_std:.4f}")

def windows_B(L, which):
    xs, ys = [], []
    for i in np.where(split == which)[0]:
        j0 = i - L
        if j0 < 0 or segment[j0] != segment[i] or split[j0] != which:
            continue
        xs.append(X_proc[j0:i])
        ys.append(y_sil[i])
    return np.stack(xs), np.asarray(ys, np.float32)

def rmse_original(pred_scaled, y_scaled):
    return float(np.sqrt(np.mean(((pred_scaled - y_scaled) * y_std) ** 2)))

In [ ]:
class SoftSensor(nn.Module):
    def __init__(self, kind, n_in=21, hidden=32):
        super().__init__()
        cell = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        kwargs = {"batch_first": True}
        if kind == "rnn":
            kwargs["nonlinearity"] = "tanh"
        self.rnn = cell(n_in, hidden, **kwargs)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1]).squeeze(-1)


def train_sensor(kind, L, epochs=20, lr=1e-4, clip=1.0):
    torch.manual_seed(0)
    sets = {name: windows_B(L, name) for name in ("train", "val", "test")}
    model = SoftSensor(kind)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(sets["train"][0]), torch.from_numpy(sets["train"][1])),
        batch_size=128, shuffle=True,
    )
    hist = []
    for _ in range(epochs):
        model.train()
        norms = []
        for xb, yb in loader:
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(xb), yb)
            loss.backward()
            norms.append(float(torch.nn.utils.clip_grad_norm_(model.parameters(), clip)))
            opt.step()
        model.eval()
        with torch.no_grad():
            tr = nn.functional.mse_loss(
                model(torch.from_numpy(sets["train"][0])), torch.from_numpy(sets["train"][1])
            ).item()
            va = nn.functional.mse_loss(
                model(torch.from_numpy(sets["val"][0])), torch.from_numpy(sets["val"][1])
            ).item()
        hist.append((tr, va, float(np.mean(norms))))
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(sets["test"][0])).numpy()
    test_rmse = rmse_original(pred, sets["test"][1])
    n_params = sum(p.numel() for p in model.parameters())
    base = rmse_original(np.zeros_like(sets["test"][1]), sets["test"][1])
    return {"hist": hist, "rmse": test_rmse, "params": n_params, "baseline": base, "n": len(sets["test"][1])}


EPOCHS_B, LR_B = 20, 1e-4
sensors = {}
for kind in ("rnn", "lstm", "gru"):
    sensors[(kind, 12)] = train_sensor(kind, 12, EPOCHS_B, LR_B)
    print(f"{kind:5} L=12  params {sensors[(kind, 12)]['params']:5d}  "
          f"test RMSE {sensors[(kind, 12)]['rmse']:.4f}")
for L in (6, 24, 48):
    for kind in ("rnn", "lstm"):
        sensors[(kind, L)] = train_sensor(kind, L, EPOCHS_B, LR_B)
        print(f"{kind:5} L={L:<2}  test RMSE {sensors[(kind, L)]['rmse']:.4f}")

print("\nPart B table, L = 12")
print(f"{'model':<8} {'params':>8} {'test RMSE':>12}")
for kind in ("rnn", "lstm", "gru"):
    row = sensors[(kind, 12)]
    print(f"{kind:<8} {row['params']:8d} {row['rmse']:12.4f}")
print(f"{'mean':<8} {'—':>8} {sensors[('rnn', 12)]['baseline']:12.4f}")

In [ ]:
fig_dir = Path("/opt/cursor/artifacts/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
for kind, color in (("rnn", "C0"), ("lstm", "C1"), ("gru", "C2")):
    hist = sensors[(kind, 12)]["hist"]
    epochs = np.arange(1, len(hist) + 1)
    ax.plot(epochs, [h[0] for h in hist], color=color, label=f"{kind} train")
    ax.plot(epochs, [h[1] for h in hist], color=color, ls="--", label=f"{kind} val")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (scaled silica)")
ax.set_title("Part B: training and validation loss, L = 12")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig(fig_dir / "partB_loss.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for kind, color in (("rnn", "C0"), ("lstm", "C1")):
    hist = sensors[(kind, 12)]["hist"]
    ax.plot(np.arange(1, len(hist) + 1), [h[2] for h in hist], color=color, label=kind)
ax.axhline(1.0, color="gray", ls=":", label="clip threshold")
ax.set_xlabel("epoch")
ax.set_ylabel("mean gradient norm before clipping")
ax.set_title("Part B: gradient norm, L = 12")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "partB_gradnorm.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
Ls = [6, 12, 24, 48]
for kind, color in (("rnn", "C0"), ("lstm", "C1")):
    ax.plot(Ls, [sensors[(kind, L)]["rmse"] for L in Ls], marker="o", color=color, label=kind)
ax.set_xlabel("window L (hours)")
ax.set_ylabel("test RMSE (% silica)")
ax.set_title("Part B: test RMSE against window length")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "partB_rmse_vs_L.png", dpi=120)
plt.show()

print("Gradient norm, epoch 1 and epoch 20")
for kind in ("rnn", "lstm"):
    h = sensors[(kind, 12)]["hist"]
    print(f"  {kind}: {h[0][2]:.3f} -> {h[-1][2]:.3f}")

In [ ]:
F_all = np.concatenate([X_proc, y_sil[:, None]], axis=1).astype(np.float32)

def windows_C(which, Tin=24, Tout=6):
    xs, y0, ys = [], [], []
    for i in np.where(split == which)[0]:
        a, b = i - Tin + 1, i + Tout
        if a < 0 or b >= n:
            continue
        if segment[a] != segment[b] or split[a] != which or split[b] != which:
            continue
        xs.append(F_all[a:i + 1])
        y0.append(y_sil[i])
        ys.append(y_sil[i + 1:i + 1 + Tout])
    return np.stack(xs), np.asarray(y0, np.float32), np.stack(ys)


class EncDec(nn.Module):
    def __init__(self, n_in=22, hidden=32, horizon=6):
        super().__init__()
        self.horizon = horizon
        self.enc = nn.LSTM(n_in, hidden, batch_first=True)
        self.dec = nn.LSTM(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x, y_last, teacher=None):
        _, (h, c) = self.enc(x)
        prev = y_last.unsqueeze(1)
        preds = []
        for t in range(self.horizon):
            out, (h, c) = self.dec(prev.unsqueeze(1), (h, c))
            yhat = self.fc(out.squeeze(1))
            preds.append(yhat)
            prev = teacher[:, t:t + 1] if teacher is not None else yhat
        return torch.cat(preds, dim=1)


class AttnEncDec(nn.Module):
    def __init__(self, n_in=22, hidden=32, horizon=6, proj=8):
        super().__init__()
        self.horizon = horizon
        self.enc = nn.LSTM(n_in, hidden, batch_first=True)
        self.Wc = nn.Linear(hidden, proj, bias=False)
        self.dec = nn.LSTM(1 + proj, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x, y_last, teacher=None):
        states, (h, c) = self.enc(x)
        s = h[-1]
        prev = y_last.unsqueeze(1)
        preds, alphas = [], []
        for t in range(self.horizon):
            scores = (states * s[:, None, :]).sum(-1)
            alpha = torch.softmax(scores, dim=1)
            context = (alpha[:, :, None] * states).sum(1)
            dec_in = torch.cat([prev, self.Wc(context)], dim=1)
            out, (h, c) = self.dec(dec_in.unsqueeze(1), (h, c))
            s = h[-1]
            yhat = self.fc(s)
            preds.append(yhat)
            alphas.append(alpha)
            prev = teacher[:, t:t + 1] if teacher is not None else yhat
        return torch.cat(preds, dim=1), torch.stack(alphas, dim=1)


def train_forecast(model, epochs, lr, clip=1.0):
    sets = {name: windows_C(name) for name in ("train", "val", "test")}
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(
        TensorDataset(*[torch.from_numpy(a) for a in sets["train"]]),
        batch_size=128, shuffle=True,
    )
    hist = []
    for _ in range(epochs):
        model.train()
        for xb, y0, yb in loader:
            opt.zero_grad()
            pred = model(xb, y0, yb)
            pred = pred[0] if isinstance(pred, tuple) else pred
            loss = nn.functional.mse_loss(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        model.eval()
        with torch.no_grad():
            def mse(name):
                batch = [torch.from_numpy(a) for a in sets[name]]
                pred = model(*batch)
                pred = pred[0] if isinstance(pred, tuple) else pred
                return nn.functional.mse_loss(pred, batch[2]).item()
            hist.append((mse("train"), mse("val")))
    return sets, hist


def step_rmse(pred, target):
    err = (pred - target) * y_std
    return np.sqrt((err ** 2).mean(axis=0))


EPOCHS_F, LR_F = 10, 1e-3
torch.manual_seed(0)
encdec = EncDec()
sets_c, hist_c = train_forecast(encdec, EPOCHS_F, LR_F)
torch.manual_seed(0)
attn = AttnEncDec()
sets_d, hist_d = train_forecast(attn, EPOCHS_F, LR_F)
print(f"encoder-decoder params {sum(p.numel() for p in encdec.parameters())}")
print(f"attention params       {sum(p.numel() for p in attn.parameters())}")
print(f"encdec  train/val MSE  {hist_c[0][0]:.3f}->{hist_c[-1][0]:.3f} / {hist_c[0][1]:.3f}->{hist_c[-1][1]:.3f}")
print(f"attn    train/val MSE  {hist_d[0][0]:.3f}->{hist_d[-1][0]:.3f} / {hist_d[0][1]:.3f}->{hist_d[-1][1]:.3f}")

encdec.eval(); attn.eval()
with torch.no_grad():
    xt, y0, yt = [torch.from_numpy(a) for a in sets_c["test"]]
    tf = encdec(xt, y0, yt).numpy()
    inf = encdec(xt, y0, None).numpy()
    xta, y0a, yta = [torch.from_numpy(a) for a in sets_d["test"]]
    tf_a, alpha_tf = attn(xta, y0a, yta)
    inf_a, alpha_inf = attn(xta, y0a, None)
    tf_a, inf_a = tf_a.numpy(), inf_a.numpy()
    alpha_inf = alpha_inf.numpy()

yt_np, y0_np = sets_c["test"][2], sets_c["test"][1]
rmse_tf = step_rmse(tf, yt_np)
rmse_inf = step_rmse(inf, yt_np)
rmse_attn = step_rmse(inf_a, sets_d["test"][2])
rmse_pers = step_rmse(np.repeat(y0_np[:, None], 6, axis=1), yt_np)

print("\nTest RMSE by forecast step")
print(f"{'step':>6} {'teacher':>10} {'inference':>10} {'attention':>10} {'persist':>10}")
for k in range(6):
    print(f"{k+1:6d} {rmse_tf[k]:10.4f} {rmse_inf[k]:10.4f} {rmse_attn[k]:10.4f} {rmse_pers[k]:10.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
steps = np.arange(1, 7)
ax.plot(steps, rmse_tf, marker="o", label="C teacher forcing")
ax.plot(steps, rmse_inf, marker="o", label="C inference")
ax.plot(steps, rmse_attn, marker="o", label="D attention inference")
ax.plot(steps, rmse_pers, marker="o", ls="--", color="gray", label="repeat last silica")
ax.set_xlabel("forecast step (hours ahead)")
ax.set_ylabel("test RMSE (% silica)")
ax.set_title("Forecast RMSE against horizon")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "forecast_rmse.png", dpi=120)
plt.show()

# one test window near the middle, inference attention
k = len(alpha_inf) // 2
heat = alpha_inf[k]
# encoder position 0 is 23 hours before the last observation
fig, ax = plt.subplots(figsize=(8, 3.2))
im = ax.imshow(heat, aspect="auto", cmap="Blues", vmin=0)
ax.set_xlabel("hours before the last observation")
ax.set_ylabel("forecast step")
ax.set_yticks(range(6), [str(i) for i in range(1, 7)])
ax.set_xticks(range(0, 24, 2), [str(-23 + i) for i in range(0, 24, 2)])
ax.set_title("Attention weights for one test window")
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout()
fig.savefig(fig_dir / "attention_heatmap.png", dpi=120)
plt.show()
print("Largest weight at each step is on encoder position", heat.argmax(1),
      "(23 = the latest hour)")
print("Mean weight on the latest hour, over this window:", heat[:, -1].mean().round(4))
print("Mean weight on the oldest hour:", heat[:, 0].mean().round(4))

In [ ]:
rnn6 = sensors[("rnn", 6)]["rmse"]
rnn48 = sensors[("rnn", 48)]["rmse"]
lstm12 = sensors[("lstm", 12)]["rmse"]
gru12 = sensors[("gru", 12)]["rmse"]
base = sensors[("rnn", 6)]["baseline"]
g0 = sensors[("rnn", 12)]["hist"][0][2]
g1 = sensors[("lstm", 12)]["hist"][0][2]
tf_lo, tf_hi = float(rmse_tf.min()), float(rmse_tf.max())

text = (
    f"Deploy the vanilla RNN soft sensor at L = 6 (test RMSE {rnn6:.2f}; training-mean "
    f"baseline {base:.2f}). The L = 12 LSTM ({lstm12:.2f}) and GRU ({gru12:.2f}) are higher. "
    f"From L = 6 to L = 48 the RNN error rises to {rnn48:.2f}, since each backward step is "
    f"scaled by at most |W_hh| tanh' and distant hours fade. The RNN gradient norm starts at "
    f"{g0:.1f}, so the clip of 1 binds; the LSTM starts at {g1:.1f}. Deploy the attention "
    f"encoder-decoder as the 6-hour forecaster. Inference RMSE at 6 hours is {rmse_attn[5]:.2f}, "
    f"versus {rmse_inf[5]:.2f} with no attention and {rmse_pers[5]:.2f} for repeating the last "
    f"silica. Teacher forcing stays between {tf_lo:.2f} and {tf_hi:.2f} while inference rises "
    f"from {rmse_inf[0]:.2f} to {rmse_inf[5]:.2f}, as each prediction is fed back. Attention "
    f"peaks on the latest hour. A VAE with epsilon = 0 on the 21 process variables should flag "
    f"hours whose reconstruction error is many times the normal value."
)
print(text)
print("\nword count", len(text.split()))
assert len(text.split()) <= 150